In [1]:
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline

In [2]:
reader = PdfReader("sample.pdf")
text = ""

for karen in reader.pages:
    text += karen.extract_text()


In [4]:
mastersplitter = RecursiveCharacterTextSplitter(
    chunk_size = 350,
    chunk_overlap = 50
)
chunks = mastersplitter.split_text(text)
print(f"Number of Chunks generated: {len(chunks)}")

Number of Chunks generated: 7


In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedd = embedding_model.encode(chunks)

print(f"Embedding successful.\nNumber of embeddings created: {len(embedd)}")

Embedding successful.
Number of embeddings created: 7


In [8]:
klient = chromadb.Client()
kollection = klient.create_collection(name="pdf-chatbot")

In [9]:
for i, chunk in enumerate(chunks):
    kollection.add(
        ids = [str(i)],
        embeddings = [embedd[i].tolist()],
        documents = [chunk]
    )

print("Collection updated successfully")

Collection updated successfully


In [10]:
llm = pipeline(
    "text2text-generation",
    model = "google/flan-t5-base"
)

Device set to use cpu


In [20]:
question = input("Ask away: ")

que = embedding_model.encode(question)

retrieved = kollection.query(
    query_embeddings = [que.tolist()],
    n_results = 3
)

Ask away:  What brand wants to sponsor the Computer Engineering Football team?


In [21]:
context = "\n\n".join(retrieved["documents"][0])

In [22]:
prompt = f"""
Answer the following question using the context below

Question:
{question}

Context:
{context}

"""

result = llm(prompt, max_length=256)
final_result = result[0]["generated_text"]
print("AI Answer:\n")
print(final_result)

Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI Answer:

YuShots Studios
